In [9]:
from torchvision import datasets, transforms as T, models
import numpy as np
from torchinfo import summary
import torch.nn as nn
import torch
from torch.utils.data import DataLoader
import torch.optim as optim

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.ToTensor(),
    T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
])
transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
])
train_ds = datasets.CIFAR100("./data/", train=True, transform=transform_train, download=True)
test_ds = datasets.CIFAR100("./data/", train=False, transform=transform_test, download=True)
print(train_ds.data.shape, test_ds.data.shape)
print(len(np.unique(test_ds.classes)))

d:\projects\Supervised-Learning-Experiments\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


(50000, 32, 32, 3) (10000, 32, 32, 3)
100


In [12]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)
summary(model)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            1,728
├─BatchNorm2d: 1-2                       (128)
├─ReLU: 1-3                              --
├─Identity: 1-4                          --
├─Sequential: 1-5                        --
│    └─BasicBlock: 2-1                   --
│    │    └─Conv2d: 3-1                  (36,864)
│    │    └─BatchNorm2d: 3-2             (128)
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  (36,864)
│    │    └─BatchNorm2d: 3-5             (128)
│    └─BasicBlock: 2-2                   --
│    │    └─Conv2d: 3-6                  (36,864)
│    │    └─BatchNorm2d: 3-7             (128)
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  (36,864)
│    │    └─BatchNorm2d: 3-10            (128)
├─Sequential: 1-6                        --
│    └─BasicBlock: 2-3                   --
│    │    └─Conv2d: 3-11     

In [13]:
from torch import optim

import torch
import copy
import torch.nn.functional as F
import matplotlib.pyplot as plt

def train(model, train_loader, test_loader, optimizer, epochs, criterion):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device, non_blocking=True)
    
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    train_losses = []
    test_losses = []
    test_accuracies = []  # List to store accuracy for each epoch
    best_accuracy = 0  # Best accuracy found
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()  # Update the learning rate

        train_losses.append(total_loss / len(train_loader))

        model.eval()
        total_test_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_test_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_loss = total_test_loss / len(test_loader)
        epoch_accuracy = correct / total
        test_losses.append(epoch_loss)
        test_accuracies.append(epoch_accuracy)

        if epoch_accuracy > best_accuracy:
            best_accuracy = epoch_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        print(f"Epoch {epoch+1}/{epochs}, Training Loss: {train_losses[-1]}, Testing Loss: {test_losses[-1]}, Testing Accuracy: {epoch_accuracy:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    print(f"Loaded the best model from epoch {best_epoch} with Testing Accuracy: {best_accuracy:.4f}")

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs+1), train_losses, label='Training Loss')
    plt.plot(range(1, epochs+1), test_losses, label='Testing Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs+1), test_accuracies, label='Testing Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

    return # train_losses, test_losses, test_accuracies, best_accuracy

In [14]:
batch_size = 128
epochs = 10

In [15]:
train_loader = DataLoader(train_ds, batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(test_ds, batch_size, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
train(model, train_loader, test_loader, optimizer, epochs, criterion)

TypeError: DataLoader is not an Optimizer

In [ ]:
for param in model.parameters():
    param.requires_grad = True

In [ ]:
batch_size = 128
epochs = 30

In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
train(model, epochs, optimizer, train_loader, test_loader, criterion, device)

Epoch: 1/30	Train Loss: 1.3697	Val Loss: 1.7793	Val Acc: 0.6919	LR: 9.76e-04
Epoch: 2/30	Train Loss: 1.3490	Val Loss: 1.7417	Val Acc: 0.7018	LR: 9.05e-04
Epoch: 3/30	Train Loss: 1.3314	Val Loss: 1.7330	Val Acc: 0.7062	LR: 7.94e-04
Epoch: 4/30	Train Loss: 1.3188	Val Loss: 1.7421	Val Acc: 0.7121	LR: 6.55e-04
Epoch: 5/30	Train Loss: 1.3071	Val Loss: 1.7369	Val Acc: 0.7099	LR: 5.00e-04
Epoch: 6/30	Train Loss: 1.3076	Val Loss: 1.7917	Val Acc: 0.6983	LR: 3.45e-04
Epoch: 7/30	Train Loss: 1.3074	Val Loss: 1.7747	Val Acc: 0.7035	LR: 2.06e-04
Epoch: 8/30	Train Loss: 1.3094	Val Loss: 1.7704	Val Acc: 0.6964	LR: 9.55e-05
Epoch: 9/30	Train Loss: 1.3014	Val Loss: 1.7311	Val Acc: 0.7093	LR: 2.45e-05
Epoch: 10/30	Train Loss: 1.3098	Val Loss: 1.7351	Val Acc: 0.7180	LR: 0.00e+00
Epoch: 11/30	Train Loss: 1.3037	Val Loss: 1.7104	Val Acc: 0.7134	LR: 2.45e-05
Epoch: 12/30	Train Loss: 1.3024	Val Loss: 1.7408	Val Acc: 0.7072	LR: 9.55e-05
Epoch: 13/30	Train Loss: 1.3020	Val Loss: 1.7718	Val Acc: 0.6992	LR: 2.06